# C2 · Self-harm specialist v2: fine-tuning on Colab

**Component 2 (hate speech and harmful language), RP-J26-DS-316.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajidush/RP-J26-DS-316/blob/c2/self-harm-v2/component2_hate_speech_detection/notebooks/self_harm_v2_colab.ipynb)

### The problem
C2's self-harm specialist (`sentinet/suicidality`, ELECTRA-base) reads violent first-person words as the writer's
own suicidality. On 19 Oct 2026 the real engine turned film lines into **urgent self-harm alerts (L4)**:

| Line | suicidality score | C2 verdict |
|---|---|---|
| "I'm going to find you, and I'm going to kill you" | 0.94 | L4 self_harm |
| "Drop the gun or I'll shoot!" | 0.94 | L4 self_harm |
| "Somebody help me! Please don't hurt me!" | 0.92 | L4 self_harm |

The model learned from suicide posts against r/teenagers, so it never saw a threat **aimed at someone else** as a
negative. HateCheck shows the same error: 23 of its 3,728 cases were labelled self_harm at age 12.

### The fix: fine-tune the specialist so "kill **you**" is not "kill **myself**"
| Training source | Public? | Role |
|---|---|---|
| Twitter suicidal intention (GitHub, laxmimerit), one of the model's own 6 training sources | yes | **replay**, so it keeps real self-harm (no forgetting) |
| Civil Comments (Borkan et al., WWW 2019; CC0), threat ≥ 0.5 | yes | real threats aimed at someone else, labelled 0 |
| Counterfactual pairs written by C2 (Kaushik et al., ICLR 2020) | written by us | "i'll kill **myself** tonight" = 1, "i'll kill **him** tonight" = 0; film commands, pleas, protective and harmless "myself" lines = 0 |

No public dataset labels short messages as self-directed vs other-directed, which is why the pairs exist. A second
recipe, **v2_public_only**, trains without them, so this run measures what they add.

**Tests, never trained on** (every training row matching one is dropped): C2's self-harm set (40 harmful, 33 benign),
C2's other-directed set (5 known film lines + 54 held-out lines written before this run), the OpenAI moderation
set (Markov et al., AAAI 2023; MIT), XSTest safe prompts (Röttger et al., NAACL 2024; CC BY 4.0) and HateCheck
(Röttger et al., ACL 2021; CC BY 4.0).

### Recipe and acceptance rule (fixed before training)
Full fine-tune from the shipped checkpoint · AdamW lr 2e-5, weight decay 0.01 · warm-up 10 % · 3 epochs · batch 32 ·
128 tokens · fp16 · gradient clipping 1.0 · seeds 316-320. Each seed keeps its best epoch on **validation**; the best
seed on validation is the candidate. Selection uses the engine's own decision point (score ≥ 0.90).

v2 replaces the shipped model only if, in C2's **full engine** at ages 9, 12 and 15
(`self_harm_model/rule.py`):
- **R1** other-directed held-out lines read as self-harm: at most half the shipped count, and ≤ 1 of the 5 known film lines
- **R2** self-harm caught on C2's self-harm sets: not lower
- **R3** self-harm lines in the other-directed set: at most 1 fewer
- **R4** benign lines alerted: not higher
- **R5** none of the 107 general cases loses a harmful alert

Then, on the laptop: ONNX parity, the INT8 gate (0 decision changes) and a full Step 9 rerun.

**Run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (about 45-60 minutes).
Nothing raw is printed: the notebook shows counts, scores and charts only.

## 1 · Machine and code

In [ ]:
BRANCH = "c2/self-harm-v2"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!rm -rf RP-J26-DS-316 && git clone -q -b $BRANCH --depth 1 https://github.com/rajidush/RP-J26-DS-316.git
%cd RP-J26-DS-316
!git log -1 --format='commit %H  %cd'
!pip install -q "transformers>=4.56" jsonschema tokenizers

In [ ]:
import hashlib, json, os, platform, subprocess, sys, time
os.environ["C2_HEAD_BACKEND"] = "torch"          # Colab has no ONNX exports: the engine runs every model in torch
import torch, transformers, datasets
from pathlib import Path
from component2_hate_speech_detection.self_harm_model import data, external, train, engine_eval, rule

RESULTS = train.RESULTS
RESULTS.mkdir(parents=True, exist_ok=True)
LOG = RESULTS / "train.log"
def log(msg):
    print(msg, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(msg + "\n")

env = {"date": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(),
       "torch": torch.__version__, "transformers": transformers.__version__, "datasets": datasets.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)",
       "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(), "branch": BRANCH}
print(json.dumps(env, indent=2))
assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU first (Runtime -> Change runtime type)."

## 2 · Test sets (downloaded first, so the training data can be checked against them)

In [ ]:
ext = external.load_all()
for name, rows in ext.items():
    groups = {}
    for r in rows:
        groups[r["group"]] = groups.get(r["group"], 0) + 1
    print(f"{name:18s} {len(rows):5d} texts, label 1 (self-harm): {sum(r['label'] for r in rows):3d}  groups: {len(groups)}")
from component2_hate_speech_detection.evaluation import other_directed_set, self_harm_set
print("C2 self-harm set   ", self_harm_set.summary())
print("C2 other-directed  ", other_directed_set.summary())

## 3 · Training data
Downloads the replay tweets (GitHub) and Civil Comments (Hugging Face, about 1.8 M comments), writes the C2 pairs,
removes every row that matches a test text (exact, or token Jaccard ≥ 0.8), and splits 15 % of each source for
validation (a pair's two sentences stay together).

In [ ]:
t0 = time.time()
test_texts = [r["text"] for rows in ext.values() for r in rows]
report = data.build(test_texts)
print(f"built in {time.time() - t0:.0f} s")
print("candidates", report["candidates"], "| unique", report["after_dedupe"], "| test texts checked", report["test_texts_checked"])
print("dropped as test overlap:", report["dropped_as_test_overlap"])
import pandas as pd
rows = []
for split in ("train", "val"):
    for key, v in report[split]["by_source"].items():
        rows.append({"split": split, "source/kind": key, "rows": v["rows"], "label 1": v["label_1"]})
display(pd.DataFrame(rows).pivot_table(index="source/kind", columns="split", values=["rows", "label 1"], aggfunc="sum").fillna(0).astype(int))
for split in ("train", "val"):
    p = data.OUT / f"{split}.jsonl"
    env[f"{split}_sha256"] = hashlib.sha256(p.read_bytes()).hexdigest()
env["data_report"] = report

In [ ]:
# What a counterfactual pair looks like (the method, shown as its template), and a few C2-written negatives.
print("pair template:  '{opener} {act} myself{tail}' -> 1     '{opener} {act} {someone else}{tail}' -> 0")
print("openers:", data.OPENERS[:4], "... acts:", data.ACTS[:5], "... targets:", data.OTHERS[:5], "...")
train_rows = data.load("train")
for kind in ("other", "film", "plea", "protective", "benign_self"):
    print(f"{kind:12s}", [r["text"] for r in train_rows if r["kind"] == kind][:3])

## 4 · Fine-tune: the shipped model's baseline, then v2 and v2_public_only (5 seeds each)
Every epoch is scored on validation; tests are scored for every seed but never steer anything.

In [ ]:
t0 = time.time()
result = train.run(external=ext, log=log)
env["training_minutes"] = round((time.time() - t0) / 60, 1)
print(f"done in {env['training_minutes']} min")

## 5 · Model-level results (score ≥ 0.90 is the engine's decision point)

In [ ]:
heads = {"shipped": result["original"]["headline"]}
for name, r in result["recipes"].items():
    heads[f"{name} (seed {r['chosen_seed']})"] = r["chosen"]["headline"]
display(pd.DataFrame(heads))
spread = {name: r["seed_spread"] for name, r in result["recipes"].items()}
keys = ["c2_other_directed_heldout.auroc", "c2_self_harm_heldout.auroc", "openai_moderation.auroc",
        "c2_other_directed_heldout.macro_f1", "openai_moderation.macro_f1"]
display(pd.DataFrame({n: {k: f"{s[k]['mean']:.3f} ± {s[k]['sd']:.3f}" for k in keys if k in s} for n, s in spread.items()}))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, r) in zip(axes, result["recipes"].items()):
    for seed in result["seeds"]:
        pts = [c for c in r["curves"] if c["seed"] == seed]
        ax.plot([c["epoch"] for c in pts], [c["val_macro_f1"] for c in pts], marker="o", label=f"seed {seed}")
    ax.axhline(result["original"]["val"]["all"]["macro_f1"], color="grey", ls="--", label="shipped model")
    ax.set_title(f"{name}: validation macro-F1 at 0.90"); ax.set_xlabel("epoch"); ax.set_xticks([1, 2, 3])
axes[0].set_ylabel("macro-F1"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS / "curves.png", dpi=120); plt.show()

In [ ]:
# Scores on the other-directed held-out set, by bucket: shipped vs v2 (each dot is one line; no text shown).
from component2_hate_speech_detection.evaluation.other_directed_set import heldout_cases
cases = heldout_cases()
shipped_m, shipped_t = train.load_model()
v2_m, v2_t = train.load_model(path=str(train.MODEL_OUT))
texts = [c.text for c in cases]
s_old, s_new = train.scores(shipped_m, shipped_t, texts), train.scores(v2_m, v2_t, texts)
buckets = list(dict.fromkeys(c.bucket for c in cases))
fig, ax = plt.subplots(figsize=(10, 4))
for i, b in enumerate(buckets):
    idx = [j for j, c in enumerate(cases) if c.bucket == b]
    ax.scatter([i - 0.12] * len(idx), [s_old[j] for j in idx], color="#c0504d", alpha=0.7, label="shipped" if i == 0 else None)
    ax.scatter([i + 0.12] * len(idx), [s_new[j] for j in idx], color="#2e75b6", alpha=0.7, label="v2" if i == 0 else None)
ax.axhline(0.90, color="grey", ls="--", lw=1); ax.text(len(buckets) - 0.5, 0.91, "engine decision point 0.90", ha="right", fontsize=8)
ax.set_xticks(range(len(buckets))); ax.set_xticklabels(buckets); ax.set_ylabel("self-harm score"); ax.legend()
ax.set_title("Other-directed held-out set: first four buckets must fall, the three self_* buckets must stay high")
plt.tight_layout(); plt.savefig(RESULTS / "other_directed_scores.png", dpi=120); plt.show()
del shipped_m, v2_m; torch.cuda.empty_cache()

## 6 · Inside C2's full engine: shipped vs v2, and the acceptance rule
The same lexicon, gates, thresholds and other three models; only the self-harm head differs. On Colab every model runs
in fp32 torch on CPU (the project is a CPU claim), so this takes a few minutes.

In [ ]:
t0 = time.time()
engine = engine_eval.run(str(train.MODEL_OUT), "torch", log=log)
verdict = rule.check(engine)
for line in ("R1", "R2", "R3", "R4", "R5"):
    print(f"{line}: {'PASS' if verdict[line]['ok'] else 'FAIL'}  {verdict[line]['detail']}")
print("\nVERDICT:", verdict["verdict"])
(RESULTS / "rule_check.json").write_text(json.dumps(verdict, indent=2) + "\n", encoding="utf-8")

In [ ]:
rows = []
for part in ("other_directed_dev", "other_directed_heldout"):
    for age in (9, 12, 15):
        for who in ("shipped", "tuned"):
            r = engine[who][part][f"age_{age}"]
            rows.append({"set": part, "age": age, "engine": who, "read as self-harm": r["other_read_as_self_harm"],
                         "still alerted": r["other_still_alerted"], "as threat": r["other_alerted_as_threat"],
                         "self-harm caught": r["self_harm_caught"]})
display(pd.DataFrame(rows).set_index(["set", "age", "engine"]))
changed = engine["general_107_cases"]["changed"]
print(f"107 general cases x 3 ages: {len(changed)} decisions differ")
display(pd.DataFrame([{"age": c["age"], "harmful": c["harmful"], "shipped": " ".join(c["shipped"]), "v2": " ".join(c["tuned"])} for c in changed]))

## 7 · Save the evidence and the model
`self_harm_v2_evidence.zip` (JSON, logs, charts; no data rows, no weights) goes into the repo under
`component2_hate_speech_detection/evaluation/results/self_harm_v2/`. `self_harm_v2_model.zip` (the v2 model, about 420 MB) stays on
the laptop for the ONNX export and the INT8 gate. Then **File → Download → .ipynb** keeps this run, with its outputs,
as the notebook evidence.

In [ ]:
(RESULTS / "env.json").write_text(json.dumps(env, indent=2) + "\n", encoding="utf-8")
!pip freeze > {RESULTS}/pip_freeze.txt
!nvidia-smi > {RESULTS}/nvidia_smi.txt
!cd component2_hate_speech_detection && zip -qr /content/self_harm_v2_evidence.zip evaluation/results/self_harm_v2
!cd component2_hate_speech_detection && zip -qr /content/self_harm_v2_model.zip models/self_harm_v2/suicidality-c2v2
!ls -lh /content/*.zip
from google.colab import files
files.download("/content/self_harm_v2_evidence.zip")
files.download("/content/self_harm_v2_model.zip")